In [ ]:
import numpy as np
from astropy.io import fits
from astropy.table import Table
from datetime import datetime
import copy
import collections
from astropy.cosmology import Planck13  # Planck (2014) Paper XVI, Table 5

import gc
gc.enable()

import sys
sys.path.append('..') 

from utils import Vcom, get_comoving_distance_h, c, z_min
# from data_stat_func import wprp, PNc
from data_stat_func import wprp, dPdN
import matplotlib.pyplot as plt


In [ ]:
z_max_arr  = np.array((       47650, 31900, 19250, 12500  ))/ c
z_max_arr_no_SGW  = np.array(19250)/ c

M_thr_arr  = np.array((-22.0, -21.0, -20.0, -19.0, -18.0, ))
C_thr_arr  = np.array([[0.0,  0.82,  0.94,  0.97,   1.2,],
                       [0.0,  0.72,  0.90,  0.94,   1.2,],
                       [0.0,  0.55,  0.85,  0.92,   1.2,],
                       [0.0,  0.50,  0.73,  0.87,   1.2,],])

M_thr_list  = [ 'Mr_22_21', 'Mr_21_20', 'Mr_20_19', 'Mr_19_18', ]
C_thr_list  = np.array([['0_p82'  , 'p82_p94', 'p94_p97', 'p97_1p2',],
                        ['0_p72'  , 'p72_p90', 'p90_p94', 'p94_1p2',],
                        ['0_p55'  , 'p55_p85', 'p85_p92', 'p92_1p2',],
                        ['0_p50'  , 'p50_p73', 'p73_p87', 'p87_1p2',],])


## $w_p(r_p)$ in box simulation

In [ ]:
### CCMD box size: 1 h^-1Gpc(comving) = 1000 h^-1Mpc

In [ ]:
from halotools.mock_observables import apply_zspace_distortion
from halotools.mock_observables import return_xyz_formatted_array
import halotools.mock_observables as htmo
# from halotools.mock_observables import wp

In [ ]:
# ###galID x y z vx vy vz Mr gmr type
# # galID -- galaxy ID (starting from 1, in a continuous ascending order)
# # x y z -- galaxy position (Mpc/h comoving)
# # vx vy vz -- galaxy peculiar velocity (km/s)
# # Mr -- r-band absolute magnitude (usual notation, K- and evolution-corrected to redshift~0.1, calculated assuming h=1)
# # gmr -- g-r color (usual notation, K- and evolution-corrected to redshift~0.1)
# # type -- CCMD model component type (0/1 - pseudo-blue/red central galaxy; 2/3 - pseudo-blue/red satellite galaxy)

# main_mock01_path = "/share/hjxu/ccmdmock/CCMD_Mock01_GalHaloInfo.fits"

# hdul = fits.open(main_mock01_path)
# # hdul[1].header
# ccmd_period = 1000 # h^-1Mpc

# ccmd_x     = hdul[1].data['x']% 1000
# ccmd_y     = hdul[1].data['y']% 1000
# ccmd_z     = hdul[1].data['z']
# ccmd_vz    = hdul[1].data['vz']

# ccmd_z     = apply_zspace_distortion(true_pos = ccmd_z, peculiar_velocity = ccmd_vz, redshift = 0, 
#                                      cosmology = Planck13, Lbox = 1000)
# ### big box size cannot use peculiar velocity ###################
# # ccmd_z     = ccmd_z + ccmd_vz/100 # v/(100h) = (v/100)/h ~ Mpc/h
# ##################################################################
# ccmd_z     = ccmd_z % 1000 # box size = 1000 Mpc/h
# ccmd_Mr    = hdul[1].data['Mr']
# ccmd_gmr   = hdul[1].data['gmr']

In [ ]:
# np.max(ccmd_z)

In [ ]:
# rp_bins = np.logspace(-0.8, 1.6, 13)
# # Comoving cylinder size in Mpc/h, cic bins, angular completeness threshold
# r_out, l_cyl = 2, 10
# cic_bin = np.concatenate([np.arange(10),
#                           np.around(np.logspace(
#                           1, np.log10(150), 30 - 10)).astype(int)]) - 0.5

# output_dict = collections.defaultdict(list)

# for j in range(4): # Mr
#     for i in range(4): # Color
#         stat = []
#         Mr_C_mask = np.logical_and(np.logical_and( M_thr_arr[j]   < ccmd_Mr , ccmd_Mr  < M_thr_arr[j+1]),
#                                    np.logical_and( C_thr_arr[j,i] < ccmd_gmr, ccmd_gmr < C_thr_arr[j,i+1]))
#     ### n_g
#         N    = np.sum(Mr_C_mask)
#         stat.append(np.array([N / 1000**3]))

#         if j == 3:
#             pi_max = 40.
#         else :
#             pi_max = 60.
#     ### wp
#         Mr_all_pos = return_xyz_formatted_array(ccmd_x, ccmd_y, ccmd_z, mask = Mr_C_mask)
#         stat.append(htmo.wp(Mr_all_pos, rp_bins, pi_max, period = ccmd_period, num_threads = 50))

#     ### dPdN
#         # Nc_out - 1, only count neighbours
#         Nc_out  =  htmo.counts_in_cylinders(Mr_all_pos, Mr_all_pos, r_out, l_cyl, period = ccmd_period, num_threads = 55) - 1 
#         # Pcic    = np.histogram(Nc_out, bins = cic_bin)[0] / len(Mr_all_pos)
#         dPdN    = np.histogram(Nc_out, bins = cic_bin, density = True)[0]

#         stat.append(dPdN)

#         output_dict['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat for item in sublist])
        
#     if j == 3:
#         np.savez('./CCMD_color_mag_Mock01_results/CCMD_Mock01_Box_Mr_C_ng_wp_dPdN_pimax60_60_60_40.npz', **output_dict)

#     print('i:',i)

In [ ]:
box = np.load('./CCMD_color_mag_Mock01_results/CCMD_Mock01_Box_Mr_C_ng_wp_dPdN_pimax60_60_60_40.npz')

In [ ]:
np.sum(box['stat_Mr_22_21_C_p97_1p2'][13:])

## $n_g, w_p(r_p), dPdN$ in Quads

In [ ]:
import pymangle
window   = pymangle.Mangle('../window_pix.dr72bright1.ply')
mask     = pymangle.Mangle('../mask_pix.dr72bright1.ply')
combmask = pymangle.Mangle('../lss_combmask_pix.dr72.ply')
combmask_info  = fits.open('../lss_combmask.dr72.fits')[1].data
mangle   = [window, mask, combmask, combmask_info]

#### $w_p(r_p)$ randoms for Quads

In [ ]:
# ### generate wp random sample for ccmd
# N       = 12000000
# ra      = np.random.random(N) * 360.0
# sin_dec = np.random.random(N) * 2 - 1
# dec     = np.arcsin(sin_dec, ) * 180.0 / np.pi

# polyid   = combmask.polyid(ra, dec)
# areamask = ((polyid != -1) & (combmask_info['FGOTMAIN'][polyid] >= 0.8) &
#              window.contains(ra, dec) & np.logical_not(mask.contains(ra, dec)))

# ra  = ra[areamask]
# dec = dec[areamask]

# # data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data

# # paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}
# # for path in paths:
# #     data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data

# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
#         data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data # CCMD Mock01
        

#         for j in range(4):
#             for i in range(4):
#                 Mr_z_C_mask  = np.logical_and(np.logical_and( M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
#                                             np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
#                                                             np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
#                 ccmdz = np.array(data['z'][Mr_z_C_mask])
#                 randz = np.tile(ccmdz, len(ra) // len(ccmdz) + 1 )[:len(ra)]
#                 # np.savez('./wp_rand_shuffle_z_CCMD/' + path + '_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i], randoms = np.vstack((ra, dec, randz)).T) 
#                 # np.savez('./wp_rand_shuffle_z_SDSS/SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i], randoms = np.vstack((ra, dec, randz)).T) 

#                 np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_' + var + '_' + quad 
#                          + '_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms', randoms = np.vstack((ra, dec, randz)).T)            
    
#         print('Observer:',var)
#         print('Quads:',quad)
#         # print('Quads:',path)
#         # print('len(ra):', len(ra))
#         # print('len(randz):',len(randz))

In [ ]:
### check shuffle z from CCMD, z_max
j = 0
i = 0
# data  = fits.open('../CCMD_main_processed/CCMD_Main_Quad0_processed_catalog.fits')[1].data
# data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data
# mag_c_mask  = np.logical_and(np.logical_and( M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                                #   np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
                                                #  np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
# randz = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_CCMD/data_randoms_Mr_22_21_C_0_p82.npz')
randz = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1/SDSS_randoms_Mr_22_21_C_0_p82.npz')
# CCMD_randz = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_CCMD/Main_Quad0_randoms_Mr_22_21_C_0_p82.npz')
CCMD_randz = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_Observer_1_Quad0_Mr_22_21_C_0_p82_randoms.npz')

# CCMDz = data['z'][mag_c_mask]

plt.hist(CCMD_randz['randoms'][:,2], histtype='step',density= True,label='shuffle z from CCMD')
plt.hist(randz['randoms'][:,2], histtype='step',density= True, label = 'shuffle z from SDSS')
plt.legend()
plt.title('Mr [-22,-21]')


### run $n_g, w_p(r_p), dPdN$

In [ ]:
# # Comoving pi_max and wp bins in Mpc/h
# rp_bins_wp = np.logspace(-0.8, 1.6, 13)
# # Comoving cylinder size in Mpc/h, cic bins, angular completeness threshold
# r_out, l_cyl = 2, 10
# cic_bin = np.concatenate([np.arange(10),
#                           np.around(np.logspace(
#                           1, np.log10(150), 30 - 10)).astype(int)]) - 0.5
# completeness_thr = 0.9

# # ### CCMD Mock01
# # paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}
# # # paths = {'SDSS_bright1'}

# # for path in paths:
# #     data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
# #     # data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
#         data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data # CCMD Mock01
#         output_dict = collections.defaultdict(list)
#     #########################################################################################################
#         for j in range(4): # Mr, z
#             for i in range(4): # Color
#                 stat = []
#                 ### Mr mask is related to the z_max 
#                 Mr_z_C_mask = np.logical_and(np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
#                                             np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr[j]),
#                                                             np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
#                 if j == 3:
#                     pi_max = 40.
#                 else :
#                     pi_max = 60.
#             ### ng
#                 N    = np.sum(Mr_z_C_mask)
#                 stat.append(np.array([N / Vcom(z_max_arr[j]),]))     
#             ### wp
#                 data_thr   = data[Mr_z_C_mask]
#                 coords_thr = np.array([np.array(data_thr['ra']),
#                                     np.array(data_thr['dec']),    
#                                     np.array(data_thr['z'])]).T
#                 # coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_CCMD/' + path + '_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
#                 # coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1/SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
#                 coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms/CCMD_' + var + '_' + quad + '_'
#                                         + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms.npz')['randoms']            

#                 stat.append(wprp(coords_thr, coords_rand_thr,
#                                 pi_max = pi_max, rp_bins = rp_bins_wp,
#                                 nthread = 55))
#             ### dPdN
#                 cen_thr = data[((data['completeness'] >= completeness_thr) *
#                                 (M_thr_arr[j] < data['Mr']) * (data['Mr'] < M_thr_arr[j+1]) *
#                                 (C_thr_arr[j,i] < data['gmr']) * ( data['gmr'] < C_thr_arr[j,i+1]) *
#                                 (get_comoving_distance_h(data['z']) >=
#                                 get_comoving_distance_h(z_min) + l_cyl) *
#                                 (get_comoving_distance_h(data['z']) <
#                                 get_comoving_distance_h(z_max_arr[j]) - l_cyl))]
#                 coords_cen_thr = np.array([np.array(cen_thr['ra']),
#                                         np.array(cen_thr['dec']),    
#                                         np.array(cen_thr['z'])]).T

#                 stat.append(dPdN(coords_cen_thr, coords_thr,
#                                 r_out = r_out,
#                                 l_cyl = l_cyl, cic_bin = cic_bin,
#                                 weigh_counts = True, 
#                                 count_weights = 1 / np.array(cen_thr['completeness'])))
                
#                 output_dict['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat for item in sublist])
                
#             # np.savez('./CCMD_color_mag_Mock01_results/CCMD_' + path + '_Mr_C_ng_wp_dPdN_pimax60_60_60_40.npz', **output_dict)
#             np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN/CCMD_' + var + '_' + quad + '_Mr_C_ng_wp_dPdN_pimax_60_60_60_40.npz', **output_dict)
#             # np.savez('./CCMD_color_mag_Mock01_results/SDSS_bright1_Mr_C_ng_wp_dPdN_pimax60_60_60_40.npz', **output_dict)

### No SGW

In [ ]:
### generate wp random sample for ccmd
N       = 12000000
ra      = np.random.random(N) * 360.0
sin_dec = np.random.random(N) * 2 - 1
dec     = np.arcsin(sin_dec, ) * 180.0 / np.pi

polyid   = combmask.polyid(ra, dec)
areamask = ((polyid != -1) & (combmask_info['FGOTMAIN'][polyid] >= 0.8) &
             window.contains(ra, dec) & np.logical_not(mask.contains(ra, dec)))

ra  = ra[areamask]
dec = dec[areamask]

# for var in range(1):
#     for quad in range(1):
#         data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data

### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data

        for j in range(4):
            for i in range(4):
                Mr_z_C_mask  = np.logical_and(np.logical_and( M_thr_arr[j]   < data['Mr'] , data['Mr']  < M_thr_arr[j+1]),
                                            np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr_no_SGW),
                                                            np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
                ccmdz = np.array(data['z'][Mr_z_C_mask])
                randz = np.tile(ccmdz, len(ra) // len(ccmdz) + 1 )[:len(ra)]
                # np.savez('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1_no_SGW/SDSS_randoms_'
                #           + M_thr_list[j] + '_C_' + C_thr_list[j,i], randoms = np.vstack((ra, dec, randz)).T)
                np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms_no_SGW/CCMD_' + var + '_' + quad
                         + '_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms', randoms = np.vstack((ra, dec, randz)).T)
    
        # print('Observer:',var)
        # print('Quads:',quad)
        # print('len(ra):', len(ra))
        # print('len(randz):',len(randz))

In [ ]:
randz = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1_no_SGW/SDSS_randoms_Mr_22_21_C_0_p82.npz')
CCMD_randz = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms_no_SGW/CCMD_Observer_1_Quad0_Mr_22_21_C_0_p82_randoms.npz')

plt.hist(CCMD_randz['randoms'][:,2], histtype='step',density= True,label='shuffle z from CCMD')
plt.hist(randz['randoms'][:,2], histtype='step',density= True, label = 'shuffle z from SDSS')
plt.legend()
plt.title('Mr [-22,-21]')


In [ ]:
# Comoving pi_max and wp bins in Mpc/h
rp_bins_wp = np.logspace(-0.8, 1.6, 13)
# Comoving cylinder size in Mpc/h, cic bins, angular completeness threshold
r_out, l_cyl = 2, 10
cic_bin = np.concatenate([np.arange(10),
                          np.around(np.logspace(
                          1, np.log10(150), 30 - 10)).astype(int)]) - 0.5
completeness_thr = 0.9

# ### CCMD Mock01
# paths = {'Main_Quad0', 'Main_Quad1', 'Main_Quad2', 'Main_Quad3',}
paths = {'SDSS_bright1'}

for path in paths:
    for path in paths:
#     data = fits.open('../CCMD_main_processed/CCMD_' + path + '_processed_catalog.fits')[1].data # CCMD Mock01
        data = fits.open('../SDSS_bright1_processed_catalog.fits')[1].data # SDSS bright1

# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
        # data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data # CCMD Mock01
        output_dict = collections.defaultdict(list)
    #########################################################################################################
        for j in range(4): # Mr, z
            for i in range(4): # Color
                stat = []
                ### Mr mask is related to the z_max 
                Mr_z_C_mask = np.logical_and(np.logical_and(M_thr_arr[j] < data['Mr'], data['Mr'] < M_thr_arr[j+1]),
                                            np.logical_and(np.logical_and(data['z'] >= z_min, data['z'] < z_max_arr_no_SGW), 
                                                            np.logical_and(C_thr_arr[j,i] < data['gmr'], data['gmr'] < C_thr_arr[j,i+1])))
                if j == 3:
                    pi_max = 40.
                else :
                    pi_max = 60.
            ### ng
                N    = np.sum(Mr_z_C_mask)
                stat.append(np.array([N / Vcom(z_max_arr[j]),]))     
            ### wp
                data_thr   = data[Mr_z_C_mask]
                coords_thr = np.array([np.array(data_thr['ra']),
                                    np.array(data_thr['dec']),    
                                    np.array(data_thr['z'])]).T
                coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/wp_rand_shuffle_z_SDSS_bright1_no_SGW/SDSS_randoms_' + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '.npz')['randoms']
                # coords_rand_thr = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_wp_randoms_no_SGW/CCMD_' + var + '_' + quad + '_'
                #                         + M_thr_list[j] + '_C_' + C_thr_list[j,i] + '_randoms.npz')['randoms']            

                stat.append(wprp(coords_thr, coords_rand_thr,
                                pi_max = pi_max, rp_bins = rp_bins_wp,
                                nthread = 55))
            ### dPdN
                cen_thr = data[((data['completeness'] >= completeness_thr) *
                                (M_thr_arr[j] < data['Mr']) * (data['Mr'] < M_thr_arr[j+1]) *
                                (C_thr_arr[j,i] < data['gmr']) * ( data['gmr'] < C_thr_arr[j,i+1]) *
                                (get_comoving_distance_h(data['z']) >=
                                get_comoving_distance_h(z_min) + l_cyl) *
                                (get_comoving_distance_h(data['z']) <
                                get_comoving_distance_h(z_max_arr[j]) - l_cyl))]
                coords_cen_thr = np.array([np.array(cen_thr['ra']),
                                        np.array(cen_thr['dec']),    
                                        np.array(cen_thr['z'])]).T

                stat.append(dPdN(coords_cen_thr, coords_thr,
                                r_out = r_out,
                                l_cyl = l_cyl, cic_bin = cic_bin,
                                weigh_counts = True, 
                                count_weights = 1 / np.array(cen_thr['completeness'])))
                
                output_dict['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat for item in sublist])
                
            # np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_' + var + '_' + quad + '_Mr_C_ng_wp_dPdN_pimax_60_60_60_40.npz', **output_dict)
            np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/SDSS_no_SGW_bright1_Mr_C_ng_wp_dPdN_pimax60_60_60_40.npz', **output_dict)

In [ ]:
# # define Bootstrap function
# def bootstrap_error(data, n_bootstrap=10000, statistic='mean'):
#     """
#     Bootstrap function
    
#     Parameters:
#     data - array like, shape = (n_measurements, n_observations) 
#     n_bootstrap - resampling times
#     statistic - ('mean', 'median', 'std')
    
#     Return:
#     bootstrap_samples - All Bootstrap samples (n_bootstrap, n_observations)
#     bootstrap_cov - Coviriance matrix (n_observations, n_observations)
#     """
#     n_measurements, n_observations = data.shape
#     bootstrap_samples = np.zeros((n_bootstrap, n_observations))
    
#     for i in range(n_bootstrap):
#         # boostraping
#         idx = np.random.choice(n_measurements, size=n_measurements, replace=True)
#         sample = data[idx, :]  # resampled data (32, 42)
        
#         # statistics
#         if statistic == 'mean':
#             bootstrap_samples[i, :] = np.mean(sample, axis=0)
#         elif statistic == 'median':
#             bootstrap_samples[i, :] = np.median(sample, axis=0)
#         elif statistic == 'std':
#             bootstrap_samples[i, :] = np.std(sample, axis=0)
#         else:
#             raise ValueError("Undefined statistics")
    
#     # covirianve matirx
#     bootstrap_cov = np.cov(bootstrap_samples, rowvar=False)  # (42, 42)
    
#     return bootstrap_samples, bootstrap_cov

In [ ]:
def cov_to_corr(input_stat, cov):
    variances = np.diag(cov) 
    zero_element = variances == 0
    zero_elements_indices = np.where(variances == 0)[0]
    input_stat = input_stat[~zero_element]
    # bestfit = bestfit[~zero_element]
    ### Remove the ith row and column
    cov = np.delete(np.delete(cov, zero_elements_indices, axis=0), zero_elements_indices, axis=1)
    ### Normalize the covariance matrix
    stdv = np.sqrt(np.diag(cov))  # Square root of the diagonal elements
    outer_diag = np.outer(stdv, stdv)   # Outer product of the diagonal elements
    corr = cov / outer_diag
    return input_stat, corr, stdv

In [ ]:
# your data 
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

output_dict_stat = collections.defaultdict(list)
output_dict_cov  = collections.defaultdict(list)

for j in range(4): # Mr, z
    for i in range(4): # Color
        data  = []
        for var in vars:
            for quad in quads:
                # temp = np.load('../CCMD_32Quads/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN/CCMD_' + var + '_' + quad + '_Mr_C_ng_wp_dPdN_pimax_60_60_60_40.npz')
                temp = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_' \
                               + var + '_' + quad + '_Mr_C_ng_wp_dPdN_pimax_60_60_60_40.npz')
                data.append(temp['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])

        data  = np.array(data)

        output_dict_stat['all_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = data
    # np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_all_samples.npz', **output_dict_stat)
    # np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_all_samples.npz', **output_dict_stat)

# ################################################################################################################
# # 16-84% error
#         percentile_16 = np.percentile(data, 16, axis=0)
#         percentile_84 = np.percentile(data, 84, axis=0)

        
#         output_dict_stat['err16_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = percentile_16
#         output_dict_cov[ 'err84_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = percentile_84

#     np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_err16.npz', **output_dict_stat)
#     np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_err84.npz', **output_dict_cov)
# ################################################################################################################


#         # Bootstrap parameters
#         # n_bootstrap = 10000  # resampling times
#         # statistic = 'mean'   # could be 'median' or 'std'
#         # bootstrap_results, cov_matrix = bootstrap_error(data, n_bootstrap, statistic)
#         # bootstrap_mean = np.mean(bootstrap_results, axis=0)  # mena value
#         # bootstrap_std = np.std(bootstrap_results, axis=0)    # stangard varianve
#         # bootstrap_ci = np.percentile(bootstrap_results, [16, 84], axis=0)  # 1 sigma - 68%

        ### calculate covariance matrix #########################################################################

        stat = []
        cov  = []
        
        original_statistics = np.mean(data, axis=0)
        cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants
        stat.append(original_statistics)
        cov.append(cov_matrix)

        output_dict_stat['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in stat for item in sublist])
        output_dict_cov[ 'cov_'  + M_thr_list[j] + '_C_' + C_thr_list[j,i]] = np.array([item for sublist in cov for item in sublist])

    # np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
    # np.savez('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_cov.npz', **output_dict_cov)
    np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_stat_mean.npz', **output_dict_stat)
    np.savez('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_cov.npz', **output_dict_cov)

In [ ]:
# ccmd_stat = np.load('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_stat_mean.npz')
# ccmd_cov_matrix = np.load('../CCMD_32Quads/CCMD_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_cov.npz')
ccmd_stat = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_stat_mean.npz')
ccmd_cov_matrix = np.load('/share/wenlk/AssemblyBias/CCMD/CCMD_32Quads_data/CCMD_32Quads_Mr_C_bins_ng_wp_dPdN_no_SGW/CCMD_no_SGW_32Quads_Mr_C_ng_wp_dPdN_pimax_60_60_60_40_cov.npz')

i = 3
j = 3
input_stat, corr_matrix, stdv = cov_to_corr(ccmd_stat['stat_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]], 
                                            ccmd_cov_matrix['cov_' + M_thr_list[j] + '_C_' + C_thr_list[j,i]])
# Plot Correlation Coviriance Matrix
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Covariance')
plt.title('Corr' + M_thr_list[j] + '_C_' + C_thr_list[j,i], fontsize = 16)
plt.show()

### CCMD 32Quads, Mr < -20, dPdN

In [ ]:
# ### generate wp random sample for ccmd
# N       = 12000000
# ra      = np.random.random(N) * 360.0
# sin_dec = np.random.random(N) * 2 - 1
# dec     = np.arcsin(sin_dec, ) * 180.0 / np.pi

# polyid   = combmask.polyid(ra, dec)
# areamask = ((polyid != -1) & (combmask_info['FGOTMAIN'][polyid] >= 0.8) &
#              window.contains(ra, dec) & np.logical_not(mask.contains(ra, dec)))

# ra  = ra[areamask]
# dec = dec[areamask]

# ### CCMD 32 Quads
# quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
# vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

# for var in vars:
#     for quad in quads:
#         data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data

#         for j in [1]:# Mr, z, Mr < -20
#             Mr_z_mask = np.logical_and(data['Mr'] < M_thr_arr[j+1], data['z'] < z_max_arr[j])
            
#             ccmdz = np.array(data['z'][Mr_z_mask])
#             randz = np.tile(ccmdz, len(ra) // len(ccmdz) + 1 )[:len(ra)]
            
#             np.savez('./CCMD_32Quads/CCMD_32Quads_Mr20_wp_randoms/CCMD_' + var + '_' + quad + '_Mrp20_randoms', randoms = np.vstack((ra, dec, randz)).T)            
    
#         print('Observer:',var)
#         print('Quads:',quad)
#         # print('len(ra):', len(ra))
#         # print('len(randz):',len(randz))

In [ ]:
SDSS_randz = np.load('/home/wenlk/AssemblyBias/wp_randoms/shuffle_z_from_sdss/sdss_randoms_shuffle_z_20p0.npz')
CCMD_randz = np.load('/home/wenlk/AssemblyBias/code/CCMD_color_mag_bin/CCMD_32Quads/CCMD_32Quads_Mr20_wp_randoms/CCMD_Observer_1_Quad0_Mrp20_randoms.npz')

plt.hist(CCMD_randz['randoms'][:,2], histtype='step',density= True,label='shuffle z from CCMD')
plt.hist(SDSS_randz['randoms'][:,2], histtype='step',density= True, label = 'shuffle z from SDSS')
plt.legend()
plt.title('Mr < -20')


In [ ]:
# Comoving pi_max and wp bins in Mpc/h
rp_bins_wp = np.logspace(-0.8, 1.6, 13)
# Comoving cylinder size in Mpc/h, cic bins, angular completeness threshold
r_out, l_cyl = 2, 10
cic_bin = np.concatenate([np.arange(10),
                          np.around(np.logspace(
                          1, np.log10(150), 30 - 10)).astype(int)]) - 0.5
completeness_thr = 0.9

### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

for var in vars:
    for quad in quads:
        data = fits.open('../CCMD_main_processed/CCMD_32Quads_processed/CCMD_' + var + '_' + quad + '_processed_catalog.fits')[1].data # CCMD Mock01
        output_dict = collections.defaultdict(list)
    #########################################################################################################
        # for j in range(4): # Mr, z
        for j in [1]: # Mr, z
            stat = []
            ### Mr mask is related to the z_max 
            Mr_z_mask = np.logical_and(data['Mr'] < M_thr_arr[j+1], data['z'] < z_max_arr[j])
                                                        
            if j == 3:
                pi_max = 40.
            else :
                pi_max = 60.
            ### ng
                N    = np.sum(Mr_z_mask)
                stat.append(np.array([N / Vcom(z_max_arr[j]),]))     
            # ### wp
            data_thr   = data[Mr_z_mask]
            coords_thr = np.array([np.array(data_thr['ra']),
                                np.array(data_thr['dec']),    
                                np.array(data_thr['z'])]).T
            coords_rand_thr = np.load('../CCMD_32Quads/CCMD_32Quads_Mr20_wp_randoms/CCMD_' + var + '_' + quad + '_Mrp20_randoms'+ '.npz')['randoms']

            stat.append(wprp(coords_thr, coords_rand_thr,
                            pi_max = pi_max, rp_bins = rp_bins_wp,
                            nthread = 55))
            ### dPdN
            cen_thr = data[((data['completeness'] >= completeness_thr) *
                            (data['Mr'] < M_thr_arr[j+1]) *
                            (get_comoving_distance_h(data['z']) <
                            get_comoving_distance_h(z_max_arr[j]) - l_cyl))]
            coords_cen_thr = np.array([np.array(cen_thr['ra']),
                                    np.array(cen_thr['dec']),    
                                    np.array(cen_thr['z'])]).T

            stat.append(dPdN(coords_cen_thr, coords_thr,
                            r_out = r_out,
                            l_cyl = l_cyl, cic_bin = cic_bin,
                            weigh_counts = True, 
                            count_weights = 1 / np.array(cen_thr['completeness'])))
            
            output_dict['stat_Mr20'] = np.array([item for sublist in stat for item in sublist])

        np.savez('../CCMD_32Quads/CCMD_32Quads_Mr20_ng_wp_dPdN/CCMD_' + var + '_' + quad + '_Mrp20_ng_wp_dPdN_pimax_60_60_60_40.npz', **output_dict)

In [ ]:
# your data 
### CCMD 32 Quads
quads = ['Quad0', 'Quad1', 'Quad2', 'Quad3', ]
vars  = ['Observer_1', 'Observer_2', 'Observer_3', 'Observer_4', 'Observer_5', 'Observer_6', 'Observer_7', 'Observer_8']

data  = []
for var in vars:
    for quad in quads:
        temp = np.load('../CCMD_32Quads/CCMD_32Quads_Mr20_ng_wp_dPdN/CCMD_' + var + '_' + quad + '_Mrp20_ng_wp_dPdN_pimax_60_60_60_40.npz')
        data.append(temp['stat_Mr20'])

data  = np.array(data)

# # Bootstrap parameters
# n_bootstrap = 10000  # resampling times
# statistic = 'mean'   # could be 'median' or 'std'

# # run bootstrap
# bootstrap_results, cov_matrix = bootstrap_error(data, n_bootstrap, statistic)

# bootstrap_mean = np.mean(bootstrap_results, axis=0)  # mena value
# bootstrap_std = np.std(bootstrap_results, axis=0)    # stangard varianve
# bootstrap_ci = np.percentile(bootstrap_results, [16, 84], axis=0)  # 1 sigma - 68%


original_statistics = np.mean(data, axis=0)
cov_matrix          = np.cov(data, rowvar=False)  # rowvar=False, column for variants
stat.append(original_statistics)
cov.append(cov_matrix)

np.save('../CCMD_32Quads/CCMD_32Quads_Mr20_ng_wp_dPdN_pimax_60_input_stat_mean', original_statistics)
np.save('../CCMD_32Quads/CCMD_32Quads_Mr20_ng_wp_dPdN_pimax_60_cov', cov_matrix)

input_stat, corr_matrix, stdv = cov_to_corr(original_statistics, cov_matrix)

# Plot Correlation Coviriance Matrix
plt.figure(figsize=(8, 6))
plt.imshow(corr_matrix, cmap='coolwarm', interpolation='nearest')
plt.colorbar(label='Covariance')
plt.title('Bootstrap Correlation Coviriance Matrix', fontsize = 16)
plt.show()